### OBJECTIVE

Decompose 'fitbit_pull.py' and develop a "from scratch" knowledge of the code. This is important for project ownership, verification, and long-term imagination. 

**Benchmark**: reliable 3-5 minute spiel on the code flow.

In [1]:
# Copied modules.

# Compiler flag that makes typing more transferrable across versions.
# __*__ is a naming convention; the full line triggers the C compiler.
from __future__ import annotations 

# Normal modules, called directly in script.
# Mainly used for data pulling and processing.
import argparse
import csv
import datetime as dt
import json
import sys
import time

# Anchors file path arithmetic to current project folder.
from pathlib import Path

# HTTP client for making API calls.
import requests

In [2]:
# Copied module-level globals. 

# In 'fitbit_pull.py', file -> __file__.
# When using python *.py, __file__ is autofilled.
file = "notebooks/pulling_data.ipynb"

# Change: resolving at the repo root.
# NOTE: ALL CAPS convention for globals.
HERE = Path().resolve().parent
SECRETS = HERE / ".secrets"
CLIENT_SECRET = SECRETS / "client_secret.json"
TOKEN_FILE = SECRETS / "token.json"
RAW_DIR = HERE / "raw"
CSV_DIR = HERE / "csv"

# Verified Google Health endpoints.
# LLM collected through API docs.
BASE = "https://health.googleapis.com/v4"
SCOPE_PREFIX = "https://www.googleapis.com/auth/googlehealth."
SCOPES = [
    SCOPE_PREFIX + "activity_and_fitness.readonly",
    SCOPE_PREFIX + "health_metrics_and_measurements.readonly",
    SCOPE_PREFIX + "sleep.readonly",
    SCOPE_PREFIX + "profile.readonly",
    SCOPE_PREFIX + "settings.readonly",
]

############################################

# Inspecting
print(HERE)
print(SECRETS)
print(CLIENT_SECRET)
# ...
# Looks fine...

/Users/markendicott/Documents/Projects/google-health-eda
/Users/markendicott/Documents/Projects/google-health-eda/.secrets
/Users/markendicott/Documents/Projects/google-health-eda/.secrets/client_secret.json


In [ ]:
# run 'python3 fitbit_pull.py auth' in terminal if token expired

In [9]:

# SIMPLEST POSSIBLE GOOGLE HEALTH API PULL
import sys; sys.path.insert(0, "..")
from fitbit_pull import _creds, BASE

# Basic user info.
requests.get(f"{BASE}/users/me/profile", 
             headers={"Authorization": f"Bearer {_creds(False).token}"}).json()


{'name': 'users/1958312538405770713/profile',
 'age': 26,
 'membershipStartDate': {'year': 2021, 'month': 12, 'day': 12},
 'userConfiguredWalkingStrideLengthMm': 747,
 'userConfiguredRunningStrideLengthMm': 1167}

In [ ]:

TYPES: dict[str, dict] = {
    "steps":                        {"kind": "interval", "rollup": True},
    "distance":                     {"kind": "interval", "rollup": True},
    "total-calories":               {"kind": "interval", "rollup": True, "max_days": 14},
    "active-minutes":               {"kind": "interval", "rollup": True, "max_days": 14},
    "floors":                       {"kind": "interval", "rollup": True},
    "active-zone-minutes":          {"kind": "interval", "rollup": False},
    "exercise":                     {"kind": "interval", "rollup": False, "page": 25},
    "sleep":                        {"kind": "sleep",    "rollup": False, "page": 25},
    "heart-rate":                   {"kind": "sample",   "rollup": False, "max_days": 1},
    "heart-rate-variability":       {"kind": "sample",   "rollup": False},
    "oxygen-saturation":            {"kind": "sample",   "rollup": False},
    "weight":                       {"kind": "sample",   "rollup": False},
    "body-fat":                     {"kind": "sample",   "rollup": False},
    "vo2-max":                      {"kind": "sample",   "rollup": False},
    "daily-resting-heart-rate":     {"kind": "daily",    "rollup": False},
    "daily-heart-rate-variability": {"kind": "daily",    "rollup": False},
    "daily-oxygen-saturation":      {"kind": "daily",    "rollup": False},
    "daily-respiratory-rate":       {"kind": "daily",    "rollup": False},
    "daily-heart-rate-zones":       {"kind": "daily",    "rollup": False},
    "daily-vo2-max":                {"kind": "daily",    "rollup": False},
}

DEFAULT_TYPES = [
    "steps", "distance", "total-calories", "active-minutes", "floors",
    "active-zone-minutes", "exercise", "sleep",
    "daily-resting-heart-rate", "daily-heart-rate-variability",
    "daily-oxygen-saturation", "daily-respiratory-rate", "weight",
]
